# tinystories-gpt4-instruct

Derives a request→story SFT dataset from `karpathy/tinystories-gpt4-clean`. Output: two parquet files in `artifacts/`.

In [1]:
import re
import string
from collections import Counter

import numpy as np
from datasets import Dataset, load_dataset

QUOTA_TRAIN = 50_000
QUOTA_VAL = 1_000
NAME_CAP = 1_000
SEED = 20260831

TEMPLATES = [
    "Tell me a story.",
    "Can you tell me a story?",
    "Tell me a story about {name}.",
    "Please tell me a story about {name}.",
    "Can you tell me a story about a {kind}?",
    "Tell me a story about a {kind} named {name}.",
    "Can you tell me a story about a {kind} named {name}?",
    "I'd like a story about a {kind} named {name}, please.",
    "Tell me a story about {name1} and {name2}.",
    "Can you tell me a story about a {kind} named {name1} and their friend {name2}?",
    "Would you tell me a story about a {kind} named {name}?",  # held out
    "How about a story about {name}?",                        # held out
]
HELD_OUT = {10, 11}

SLOTS = [
    {field for _, field, _, _ in string.Formatter().parse(t) if field}
    for t in TEMPLATES
]

/Users/jefferyharrell/Pondside/Workshop/Projects/tinystories-gpt4-instruct/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("karpathy/tinystories-gpt4-clean", split="train")
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2732634
})


In [3]:
NAMED_RE = re.compile(r"\b(?:a|an|the) (?:\w+ ){0,2}?(\w+) named ([A-Z]\w+)")


def extract(text):
    names, kinds = [], {}
    for kind, name in NAMED_RE.findall(text):
        if name not in names:
            names.append(name)
            kinds[name] = kind
    return names, kinds


for i in [0, 1, 3, 12]:
    text = dataset[i]["text"]
    print(f"{i}: {extract(text)}  --  {text[:60]}...")

0: (['Jack'], {'Jack': 'boy'})  --  Once there was a little boy named Jack. He was only three ye...
1: ([], {})  --  One day, Tom and Sue went to the park. They wanted to play a...
3: (['Lily'], {'Lily': 'girl'})  --  Once upon a time, in a small town, there was a kind girl nam...
12: ([], {})  --  Once upon a time there was a tomato. The tomato was ready to...


In [4]:
census = Counter()
for batch in dataset.iter(batch_size=10_000):
    for text in batch["text"]:
        names, _ = extract(text)
        census.update(names)

print(f"{sum(census.values()):,} name occurrences, {len(census):,} distinct names")
for name, count in census.most_common(20):
    print(f"{count:8,}  {name}")

1,964,440 name occurrences, 4,271 distinct names
 541,048  Tim
 221,963  Lily
 161,772  Tom
 136,031  Sue
 104,896  Max
  76,991  Lucy
  76,023  Spot
  62,425  Mia
  51,611  Amy
  44,706  Sam
  42,305  Bob
  25,913  Kitty
  23,884  Sally
  18,182  Fluffy
  15,211  Bobo
  14,424  Ben
  12,750  Fin
  10,838  Timmy
  10,138  Jane
   9,390  Buddy


In [5]:
def make_pair(rng, source_id, text, template_ids):
    names, kinds = extract(text)
    if not names:
        return None
    have = set()
    if len(names) >= 1:
        have |= {"name", "kind"}
    if len(names) >= 2:
        have |= {"name1", "name2"}

    eligible = [t for t in template_ids if SLOTS[t] <= have]
    if not eligible:
        return None
    weights = np.array([1.0 if not SLOTS[t] else 4.0 for t in eligible])
    template_id = int(rng.choice(eligible, p=weights / weights.sum()))

    fills = {}
    if names:
        fills = {"name": names[0], "kind": kinds[names[0]], "name1": names[0]}
    if len(names) >= 2:
        fills["name2"] = names[1]

    used = sorted(SLOTS[template_id])
    return {
        "prompt": TEMPLATES[template_id].format(**{k: fills[k] for k in used}),
        "story": text,
        "source_id": source_id,
        "template_id": template_id,
        "names": [fills[k] for k in used if k.startswith("name")],
        "kind": fills.get("kind", "") if "kind" in used else "",
        "all_names": names,
    }

In [6]:
rng = np.random.default_rng(SEED)
train_ids = [t for t in range(len(TEMPLATES)) if t not in HELD_OUT]
all_ids = list(range(len(TEMPLATES)))

validation, train = [], []
name_counts = Counter()
scanned = 0

for batch_start, batch in enumerate(dataset.iter(batch_size=10_000)):
    for offset, text in enumerate(batch["text"]):
        source_id = batch_start * 10_000 + offset
        scanned += 1

        if len(validation) < QUOTA_VAL:
            pair = make_pair(rng, source_id, text, all_ids)
            if pair:
                validation.append(pair)
            continue

        pair = make_pair(rng, source_id, text, train_ids)
        if pair is None:
            continue
        protagonist = pair["all_names"][0]
        if name_counts[protagonist] >= NAME_CAP:
            continue
        name_counts[protagonist] += 1
        train.append(pair)
        if len(train) >= QUOTA_TRAIN:
            break
    if len(train) >= QUOTA_TRAIN:
        break

print(f"scanned {scanned:,} stories -> {len(validation):,} validation, {len(train):,} train")

scanned 381,599 stories -> 1,000 validation, 50,000 train


In [7]:
print("train pairs per template:")
for tid, count in sorted(Counter(p["template_id"] for p in train).items()):
    print(f"  {tid:2d}  {count:6,}  {TEMPLATES[tid]}")
print()
print("validation pairs per template:")
for tid, count in sorted(Counter(p["template_id"] for p in validation).items()):
    held = "  (held out)" if tid in HELD_OUT else ""
    print(f"  {tid:2d}  {count:6,}  {TEMPLATES[tid]}{held}")
print()
print("most-requested names in train after the cap:")
for name, count in Counter(n for p in train for n in p["names"]).most_common(10):
    print(f"{count:8,}  {name}")

train pairs per template:
   0   1,830  Tell me a story.
   1   1,808  Can you tell me a story?
   2   7,351  Tell me a story about {name}.
   3   7,270  Please tell me a story about {name}.
   4   7,137  Can you tell me a story about a {kind}?
   5   7,199  Tell me a story about a {kind} named {name}.
   6   7,378  Can you tell me a story about a {kind} named {name}?
   7   7,144  I'd like a story about a {kind} named {name}, please.
   8   1,442  Tell me a story about {name1} and {name2}.
   9   1,441  Can you tell me a story about a {kind} named {name1} and their friend {name2}?

validation pairs per template:
   0      32  Tell me a story.
   1      30  Can you tell me a story?
   2     111  Tell me a story about {name}.
   3      96  Please tell me a story about {name}.
   4     103  Can you tell me a story about a {kind}?
   5     111  Tell me a story about a {kind} named {name}.
   6     106  Can you tell me a story about a {kind} named {name}?
   7     112  I'd like a story abo

In [8]:
Dataset.from_list(train).to_parquet("artifacts/train.parquet")
Dataset.from_list(validation).to_parquet("artifacts/validation.parquet")

for pair in train[:3]:
    print(f"[{pair['source_id']}] {pair['prompt']}")
    print(f"    {pair['story'][:80]}...")
    print()

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 13.24ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 583.03ba/s]

[1682] I'd like a story about a boy named Tim, please.
    One day, a little boy named Tim was eager to play outside. He saw that the sun w...

[1686] Please tell me a story about Lily.
    Once upon a time, there was a little girl named Lily. Lily lived in a small hous...

[1688] Can you tell me a story about a boy named Tim?
    One day, a little boy named Tim found a big guitar. He was very happy. Tim loved...

